# 05_data_analysis

En este notebook se responden las preguntas de negocio usando la tabla `ANALYTICS.OBT_TRIPS` con Spark.

Las consultas se parametrizan con:
- años
- meses
- servicios

Además, la conexión usa variables de ambiente para Snowflake.

## Configuración y conexión

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession

In [2]:
YEARS = list(range(2015, 2026))
MONTHS = list(range(1, 13))
SERVICES = ["yellow", "green"]

DATABASE = os.getenv("SNOWFLAKE_DATABASE")
ANALYTICS_SCHEMA = os.getenv("SNOWFLAKE_SCHEMA_ANALYTICS")
OBT_TABLE = "OBT_TRIPS"

assert DATABASE is not None, "Falta SNOWFLAKE_DATABASE"
assert ANALYTICS_SCHEMA is not None, "Falta SNOWFLAKE_SCHEMA_ANALYTICS"

In [3]:
try:
    spark.stop()
except:
    pass

spark = (
    SparkSession.builder
    .appName("P3 Data Analysis")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        ",".join([
            "net.snowflake:spark-snowflake_2.12:2.16.0-spark_3.4",
            "net.snowflake:snowflake-jdbc:3.13.30"
        ])
    )
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.timestampType", "TIMESTAMP_LTZ")
    .getOrCreate()
)

spark

In [4]:
sfOptionsAnalytics = {
    "sfURL": f"{os.getenv('SNOWFLAKE_ACCOUNT')}.snowflakecomputing.com",
    "sfUser": os.getenv("SNOWFLAKE_USER"),
    "sfPassword": os.getenv("SNOWFLAKE_PASSWORD"),
    "sfDatabase": DATABASE,
    "sfSchema": ANALYTICS_SCHEMA,
    "sfWarehouse": os.getenv("SNOWFLAKE_WAREHOUSE"),
    "sfRole": os.getenv("SNOWFLAKE_ROLE"),
}

In [5]:
services_sql = ", ".join([f"'{s}'" for s in SERVICES])
years_sql = ", ".join([str(y) for y in YEARS])
months_sql = ", ".join([str(m) for m in MONTHS])

query = f"""
SELECT *
FROM {ANALYTICS_SCHEMA}.{OBT_TABLE}
WHERE source_year IN ({years_sql})
  AND source_month IN ({months_sql})
  AND service_type IN ({services_sql})
"""

obt = (
    spark.read
    .format("snowflake")
    .options(**sfOptionsAnalytics)
    .option("query", query)
    .load()
)

obt.createOrReplaceTempView("obt_trips")
print("Schema loaded:")
obt.printSchema()

Schema loaded:
root
 |-- PICKUP_DATETIME: timestamp (nullable = true)
 |-- DROPOFF_DATETIME: timestamp (nullable = true)
 |-- PICKUP_DATE: date (nullable = true)
 |-- PICKUP_HOUR: decimal(38,0) (nullable = true)
 |-- DROPOFF_DATE: date (nullable = true)
 |-- DROPOFF_HOUR: decimal(38,0) (nullable = true)
 |-- DAY_OF_WEEK: decimal(38,0) (nullable = true)
 |-- MONTH: decimal(38,0) (nullable = true)
 |-- YEAR: decimal(38,0) (nullable = true)
 |-- PU_LOCATION_ID: decimal(38,0) (nullable = true)
 |-- PU_ZONE: string (nullable = true)
 |-- PU_BOROUGH: string (nullable = true)
 |-- DO_LOCATION_ID: decimal(38,0) (nullable = true)
 |-- DO_ZONE: string (nullable = true)
 |-- DO_BOROUGH: string (nullable = true)
 |-- SERVICE_TYPE: string (nullable = true)
 |-- VENDOR_ID: decimal(38,0) (nullable = true)
 |-- VENDOR_NAME: string (nullable = true)
 |-- RATE_CODE_ID: decimal(38,0) (nullable = true)
 |-- RATE_CODE_DESC: string (nullable = true)
 |-- PAYMENT_TYPE: decimal(38,0) (nullable = true)
 |-- PA

## Pregunta a. Top 10 zonas de pickup por volumen mensual

In [7]:
q_a = spark.sql("""
WITH ranked AS (
    SELECT
        source_year,
        source_month,
        pu_zone,
        COUNT(*) AS total_trips,
        ROW_NUMBER() OVER (
            PARTITION BY source_year, source_month
            ORDER BY COUNT(*) DESC
        ) AS rn
    FROM obt_trips
    GROUP BY source_year, source_month, pu_zone
)
SELECT
    source_year,
    source_month,
    pu_zone,
    total_trips
FROM ranked
WHERE rn <= 10
ORDER BY source_year, source_month, total_trips DESC
""")

q_a.show(100, truncate=False)

+-----------+------------+----------------------------+-----------+
|source_year|source_month|pu_zone                     |total_trips|
+-----------+------------+----------------------------+-----------+
|2017       |10          |Upper East Side South       |403810     |
|2017       |10          |Midtown Center              |374521     |
|2017       |10          |Upper East Side North       |362592     |
|2017       |10          |Penn Station/Madison Sq West|341463     |
|2017       |10          |Midtown East                |333392     |
|2017       |10          |Times Sq/Theatre District   |324601     |
|2017       |10          |Union Sq                    |317623     |
|2017       |10          |Murray Hill                 |309129     |
|2017       |10          |Clinton East                |299307     |
|2017       |10          |Lincoln Square East         |297423     |
|2017       |11          |Upper East Side South       |397322     |
|2017       |11          |Midtown Center        

**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta b. Top 10 zonas de dropoff por volumen mensual

In [9]:
q_b = spark.sql("""
WITH ranked AS (
    SELECT
        source_year,
        source_month,
        do_zone,
        COUNT(*) AS total_trips,
        ROW_NUMBER() OVER (
            PARTITION BY source_year, source_month
            ORDER BY COUNT(*) DESC
        ) AS rn
    FROM obt_trips
    GROUP BY source_year, source_month, do_zone
)
SELECT
    source_year,
    source_month,
    do_zone,
    total_trips
FROM ranked
WHERE rn <= 10
ORDER BY source_year, source_month, total_trips DESC
""")

q_b.show(100, truncate=False)

+-----------+------------+----------------------------+-----------+
|source_year|source_month|do_zone                     |total_trips|
+-----------+------------+----------------------------+-----------+
|2017       |10          |Upper East Side North       |390856     |
|2017       |10          |Midtown Center              |370519     |
|2017       |10          |Upper East Side South       |361133     |
|2017       |10          |Murray Hill                 |313914     |
|2017       |10          |Midtown East                |302666     |
|2017       |10          |Times Sq/Theatre District   |299519     |
|2017       |10          |Union Sq                    |276058     |
|2017       |10          |Lincoln Square East         |268592     |
|2017       |10          |Penn Station/Madison Sq West|266828     |
|2017       |10          |Clinton East                |263385     |
|2017       |11          |Upper East Side North       |386556     |
|2017       |11          |Midtown Center        

**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta c. Evolución mensual de total_amount y tip_pct por borough

In [10]:
q_c = spark.sql("""
SELECT
    source_year,
    source_month,
    pu_borough,
    SUM(total_amount) AS total_amount_sum,
    AVG(tip_pct) AS avg_tip_pct
FROM obt_trips
GROUP BY source_year, source_month, pu_borough
ORDER BY source_year, source_month, pu_borough
""")
q_c.show(100, truncate=False)

+-----------+------------+-------------+----------------+-------------+
|source_year|source_month|pu_borough   |total_amount_sum|avg_tip_pct  |
+-----------+------------+-------------+----------------+-------------+
|2016       |5           |Queens       |36798857.17     |13.506632397 |
|2016       |5           |Staten Island|22419.2         |23.849066959 |
|2016       |5           |Unknown      |2281341.55      |15.511081101 |
|2016       |5           |null         |233423.5        |469.931256927|
|2016       |6           |Bronx        |1119088.52      |4.541982665  |
|2016       |6           |Brooklyn     |12392737.91     |13.884329747 |
|2016       |6           |EWR          |20498.64        |141.20092511 |
|2016       |6           |Manhattan    |154307693.6     |15.026991833 |
|2016       |6           |Queens       |35046217.88     |16.476344702 |
|2016       |6           |Staten Island|25607.86        |15.005022727 |
|2016       |6           |Unknown      |2128963.68      |15.5033

**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta d. Ticket promedio (avg total_amount) por service_type y mes

In [11]:
q_d = spark.sql("""
SELECT
    source_year,
    source_month,
    service_type,
    AVG(total_amount) AS avg_ticket
FROM obt_trips
GROUP BY source_year, source_month, service_type
ORDER BY source_year, source_month, service_type
""")
q_d.show(100, truncate=False)

+-----------+------------+------------+------------+
|source_year|source_month|service_type|avg_ticket  |
+-----------+------------+------------+------------+
|2016       |5           |yellow      |16.600902583|
|2016       |6           |green       |15.078434506|
|2016       |6           |yellow      |16.670237898|
|2016       |7           |green       |14.960827589|
|2016       |7           |yellow      |16.421501281|
|2016       |8           |green       |14.872124641|
|2016       |8           |yellow      |16.326383662|
|2016       |9           |green       |14.923286793|
|2016       |9           |yellow      |16.961342156|
|2016       |10          |green       |14.621574823|
|2016       |10          |yellow      |16.552242145|
|2016       |11          |green       |14.272024259|
|2016       |11          |yellow      |16.552577056|
|2016       |12          |green       |14.003471692|
|2016       |12          |yellow      |16.164914862|
|2017       |1           |green       |13.6513

**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta e. Viajes por hora del día y día de semana (picos)

In [12]:
q_e = spark.sql("""
SELECT
    pickup_hour,
    day_of_week,
    COUNT(*) AS total_trips
FROM obt_trips
GROUP BY pickup_hour, day_of_week
ORDER BY day_of_week, pickup_hour
""")
q_e.show(200, truncate=False)

+-----------+-----------+-----------+
|pickup_hour|day_of_week|total_trips|
+-----------+-----------+-----------+
|18         |1          |7603117    |
|19         |1          |6981221    |
|20         |1          |6363722    |
|21         |1          |6070592    |
|22         |1          |5168127    |
|23         |1          |3650815    |
|0          |2          |2288001    |
|1          |2          |1275910    |
|2          |2          |765475     |
|3          |2          |515384     |
|4          |2          |573392     |
|5          |2          |1100365    |
|6          |2          |2940457    |
|7          |2          |5302269    |
|8          |2          |6563384    |
|9          |2          |6320439    |
|10         |2          |5875120    |
|11         |2          |5901399    |
|12         |2          |6170074    |
|13         |2          |6212662    |
|14         |2          |6709433    |
|12         |3          |6410409    |
|13         |3          |6438002    |
|14         

**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta f. p50/p90 de trip_duration_min por borough de pickup

In [15]:
q_f = (
    spark.read
    .format("snowflake")
    .options(**sfOptionsAnalytics)
    .option(
        "query",
        f"""
        SELECT
            pu_borough,
            APPROX_PERCENTILE(trip_duration_min, 0.5) AS p50_trip_duration_min,
            APPROX_PERCENTILE(trip_duration_min, 0.9) AS p90_trip_duration_min
        FROM {OBT_TABLE}
        WHERE pu_borough IS NOT NULL
          AND trip_duration_min IS NOT NULL
          AND trip_duration_min BETWEEN 0 AND 300
        GROUP BY pu_borough
        ORDER BY pu_borough
        """
    )
    .load()
)

q_f.show(100, truncate=False)

+-------------+---------------------+---------------------+
|PU_BOROUGH   |P50_TRIP_DURATION_MIN|P90_TRIP_DURATION_MIN|
+-------------+---------------------+---------------------+
|Bronx        |13.993766893         |38.856107867         |
|EWR          |0.5                  |25.130685829         |
|Brooklyn     |13.151425527         |32.651795118         |
|Staten Island|27.303013292         |71.154774068         |
|Unknown      |11.236313016         |28.707435632         |
|Manhattan    |10.875292744         |25.051610125         |
|Queens       |25.068190101         |54.533752007         |
+-------------+---------------------+---------------------+



**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta g. avg_speed_mph por franja horaria (6–9, 17–20) y borough

In [16]:
q_g = spark.sql("""
SELECT
    CASE
        WHEN pickup_hour BETWEEN 6 AND 9 THEN '06-09'
        WHEN pickup_hour BETWEEN 17 AND 20 THEN '17-20'
        ELSE 'other'
    END AS franja_horaria,
    pu_borough,
    AVG(avg_speed_mph) AS avg_speed_mph
FROM obt_trips
WHERE pickup_hour BETWEEN 6 AND 9
   OR pickup_hour BETWEEN 17 AND 20
GROUP BY
    CASE
        WHEN pickup_hour BETWEEN 6 AND 9 THEN '06-09'
        WHEN pickup_hour BETWEEN 17 AND 20 THEN '17-20'
        ELSE 'other'
    END,
    pu_borough
ORDER BY franja_horaria, pu_borough
""")
q_g.show(100, truncate=False)

+--------------+-------------+-------------+
|franja_horaria|pu_borough   |avg_speed_mph|
+--------------+-------------+-------------+
|17-20         |Bronx        |50.373466505 |
|17-20         |Brooklyn     |25.093705225 |
|06-09         |Unknown      |38.555005632 |
|06-09         |null         |638.382745281|
|17-20         |EWR          |775.052715618|
|17-20         |Manhattan    |38.53549458  |
|06-09         |Queens       |362.75134039 |
|06-09         |Staten Island|95.623081092 |
|06-09         |Bronx        |143.303020737|
|06-09         |Brooklyn     |54.720639064 |
|06-09         |EWR          |767.108925988|
|06-09         |Manhattan    |22.119857451 |
|17-20         |Queens       |56.270565632 |
|17-20         |Staten Island|128.500423729|
|17-20         |Unknown      |19.448814861 |
|17-20         |null         |601.827515097|
+--------------+-------------+-------------+



**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta h. Participación por payment_type_desc y su relación con tip_pct

In [17]:
q_h = spark.sql("""
SELECT
    payment_type_desc,
    COUNT(*) AS total_trips,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_share,
    AVG(tip_pct) AS avg_tip_pct
FROM obt_trips
GROUP BY payment_type_desc
ORDER BY total_trips DESC
""")
q_h.show(100, truncate=False)

+-----------------+-----------+---------+---------------+
|payment_type_desc|total_trips|pct_share|avg_tip_pct    |
+-----------------+-----------+---------+---------------+
|Unknown          |21243126   |2.48     |5.37966327     |
|Cash             |253858150  |29.59    |0.0009490730162|
|Credit card      |577649581  |67.34    |23.497046379   |
|Dispute          |2100334    |0.24     |0.05700057226  |
|No charge        |2931565    |0.34     |0.04425379715  |
+-----------------+-----------+---------+---------------+



**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta i. ¿Qué rate_code_desc concentran mayor trip_distance y total_amount?

In [18]:
q_i = spark.sql("""
SELECT
    rate_code_desc,
    SUM(trip_distance) AS total_trip_distance,
    SUM(total_amount) AS total_amount_sum
FROM obt_trips
GROUP BY rate_code_desc
ORDER BY total_trip_distance DESC, total_amount_sum DESC
""")
q_i.show(100, truncate=False)

+------------------+-------------------+----------------+
|rate_code_desc    |total_trip_distance|total_amount_sum|
+------------------+-------------------+----------------+
|Unknown           |869274636.04       |615488892.44    |
|JFK               |434573973.14       |1389337244.33   |
|Standard rate     |3563266485.18      |13565493203.52  |
|Negotiated fare   |92039921.99        |197640822.56    |
|Newark            |28899696.81        |164091986.33    |
|Nassau/Westchester|21245158.57        |75275577.21     |
|Group ride        |8941.68            |75310.41        |
+------------------+-------------------+----------------+



**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta j. Mix yellow vs green por mes y borough

In [19]:
q_j = spark.sql("""
SELECT
    source_year,
    source_month,
    pu_borough,
    service_type,
    COUNT(*) AS total_trips,
    ROUND(
        100.0 * COUNT(*) /
        SUM(COUNT(*)) OVER (PARTITION BY source_year, source_month, pu_borough),
        2
    ) AS pct_mix
FROM obt_trips
GROUP BY source_year, source_month, pu_borough, service_type
ORDER BY source_year, source_month, pu_borough, service_type
""")
q_j.show(200, truncate=False)

+-----------+------------+-------------+------------+-----------+-------+
|source_year|source_month|pu_borough   |service_type|total_trips|pct_mix|
+-----------+------------+-------------+------------+-----------+-------+
|2015       |1           |Bronx        |green       |87343      |90.66  |
|2015       |1           |Bronx        |yellow      |9000       |9.34   |
|2015       |1           |Brooklyn     |green       |564018     |71.30  |
|2015       |1           |Brooklyn     |yellow      |226994     |28.70  |
|2015       |1           |EWR          |green       |13         |5.02   |
|2015       |1           |EWR          |yellow      |246        |94.98  |
|2015       |1           |Manhattan    |green       |426358     |3.55   |
|2015       |1           |Manhattan    |yellow      |11569730   |96.45  |
|2015       |1           |Queens       |green       |404153     |39.23  |
|2015       |1           |Queens       |yellow      |625971     |60.77  |
|2015       |1           |Staten Islan

**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta k. Top 20 flujos PU→DO por volumen y su ticket promedio

In [20]:
q_k = spark.sql("""
SELECT
    pu_zone,
    do_zone,
    COUNT(*) AS total_trips,
    AVG(total_amount) AS avg_ticket
FROM obt_trips
GROUP BY pu_zone, do_zone
ORDER BY total_trips DESC
LIMIT 20
""")
q_k.show(20, truncate=False)

+---------------------------------------------+--------------------------------+-----------+------------+
|pu_zone                                      |do_zone                         |total_trips|avg_ticket  |
+---------------------------------------------+--------------------------------+-----------+------------+
|Port Richmond                                |Westerleigh                     |23         |15.329565217|
|Woodlawn/Wakefield                           |Bushwick North                  |23         |62.715652174|
|Newark Airport                               |Union Sq                        |23         |98.805217391|
|Fresh Meadows                                |Midtown South                   |23         |51.569130435|
|Erasmus                                      |Whitestone                      |23         |89.932173913|
|Hammels/Arverne                              |Douglaston                      |23         |55.753478261|
|Bayside                                      

**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta l. Distribución de passenger_count y efecto en total_amount

In [21]:
q_l = spark.sql("""
SELECT
    passenger_count,
    COUNT(*) AS total_trips,
    AVG(total_amount) AS avg_total_amount
FROM obt_trips
GROUP BY passenger_count
ORDER BY passenger_count
""")
q_l.show(100, truncate=False)

+---------------+-----------+----------------+
|passenger_count|total_trips|avg_total_amount|
+---------------+-----------+----------------+
|0              |5763113    |19.574928784    |
|1              |610390360  |18.278442857    |
|2              |117994181  |19.67979146     |
|3              |32660373   |19.078618465    |
|4              |15791942   |20.015769468    |
|5              |33400593   |17.051933387    |
|6              |20537061   |16.89112264     |
|7              |1728       |50.800949074    |
|8              |1873       |58.180832888    |
|9              |994        |74.074818913    |
|96             |2          |12.59           |
|112            |1          |14.76           |
|32             |1          |60.35           |
|48             |1          |40.3            |
|192            |2          |9.15            |
|null           |21240531   |26.345499743    |
+---------------+-----------+----------------+



**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta m. Impacto de tolls_amount y congestion_surcharge por zona

In [22]:
q_m = spark.sql("""
SELECT
    pu_zone,
    AVG(tolls_amount) AS avg_tolls_amount,
    AVG(congestion_surcharge) AS avg_congestion_surcharge,
    SUM(total_amount) AS total_amount_sum
FROM obt_trips
GROUP BY pu_zone
ORDER BY total_amount_sum DESC
""")
q_m.show(100, truncate=False)

+---------------------------------------------+----------------+------------------------+----------------+
|pu_zone                                      |avg_tolls_amount|avg_congestion_surcharge|total_amount_sum|
+---------------------------------------------+----------------+------------------------+----------------+
|Van Cortlandt Park                           |0.8994265367    |0.08128704488           |375080.81       |
|East Flushing                                |0.6233378591    |0.09195688226           |369242.74       |
|Charleston/Tottenville                       |7.231675722     |0.008183306056          |326246.1        |
|Pelham Bay Park                              |0.4299291814    |0.1541626331            |317747.38       |
|Marine Park/Floyd Bennett Field              |1.97399748      |0.2214867617            |281509.31       |
|City Island                                  |1.794740219     |0.05407047388           |266831.93       |
|Mariners Harbor                     

**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta n. Proporción de viajes cortos vs largos por borough y estacionalidad

In [23]:
q_n = spark.sql("""
SELECT
    pu_borough,
    source_year,
    source_month,
    CASE
        WHEN trip_distance < 2 THEN 'short'
        WHEN trip_distance >= 2 AND trip_distance < 10 THEN 'medium'
        ELSE 'long'
    END AS trip_length_bucket,
    COUNT(*) AS total_trips
FROM obt_trips
GROUP BY
    pu_borough,
    source_year,
    source_month,
    CASE
        WHEN trip_distance < 2 THEN 'short'
        WHEN trip_distance >= 2 AND trip_distance < 10 THEN 'medium'
        ELSE 'long'
    END
ORDER BY pu_borough, source_year, source_month, trip_length_bucket
""")
q_n.show(200, truncate=False)

+----------+-----------+------------+------------------+-----------+
|pu_borough|source_year|source_month|trip_length_bucket|total_trips|
+----------+-----------+------------+------------------+-----------+
|Bronx     |2015       |1           |long              |3226       |
|Bronx     |2015       |1           |medium            |43107      |
|Bronx     |2015       |1           |short             |50010      |
|Bronx     |2015       |2           |long              |3860       |
|Bronx     |2015       |2           |medium            |57943      |
|Bronx     |2015       |2           |short             |67425      |
|Bronx     |2015       |3           |long              |4579       |
|Bronx     |2015       |3           |medium            |66913      |
|Bronx     |2015       |3           |short             |74027      |
|Bronx     |2015       |4           |long              |4672       |
|Bronx     |2015       |4           |medium            |58285      |
|Bronx     |2015       |4         

**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta o. Diferencias por vendor en avg_speed_mph y trip_duration_min

In [24]:
q_o = spark.sql("""
SELECT
    vendor_name,
    AVG(avg_speed_mph) AS avg_speed_mph,
    AVG(trip_duration_min) AS avg_trip_duration_min
FROM obt_trips
GROUP BY vendor_name
ORDER BY avg_speed_mph DESC
""")
q_o.show(100, truncate=False)

+----------------------------+-------------+---------------------+
|vendor_name                 |avg_speed_mph|avg_trip_duration_min|
+----------------------------+-------------+---------------------+
|Unknown                     |10.73408348  |14.30829253          |
|Creative Mobile Technologies|69.635075574 |17.014308659         |
|Helix                       |null         |0                    |
|Myle Technologies           |702.42304153 |45.399758254         |
|Curb Mobility               |17.852271694 |21.147784949         |
+----------------------------+-------------+---------------------+



**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta p. Relación método de pago ↔ tip_amount por hora

In [25]:
q_p = spark.sql("""
SELECT
    pickup_hour,
    payment_type_desc,
    AVG(tip_amount) AS avg_tip_amount,
    AVG(tip_pct) AS avg_tip_pct
FROM obt_trips
GROUP BY pickup_hour, payment_type_desc
ORDER BY pickup_hour, payment_type_desc
""")
q_p.show(200, truncate=False)

+-----------+-----------------+---------------+---------------+
|pickup_hour|payment_type_desc|avg_tip_amount |avg_tip_pct    |
+-----------+-----------------+---------------+---------------+
|0          |Cash             |0.0001032178569|0.0007325951425|
|0          |Credit card      |3.017361692    |23.917689893   |
|0          |Dispute          |0.008555548578 |0.06698249711  |
|0          |No charge        |0.002396423806 |0.01510906443  |
|0          |Unknown          |0.6383550223   |3.632519219    |
|1          |Cash             |9.0451582e-05  |0.0006237355347|
|1          |Credit card      |2.789983771    |24.311540865   |
|1          |Dispute          |0.005554109114 |0.03640193796  |
|1          |No charge        |0.003981474705 |0.03559538268  |
|1          |Unknown          |0.5943767818   |3.39944909     |
|2          |Cash             |9.434705892e-05|0.0006815719675|
|2          |Credit card      |2.642249843    |24.177677557   |
|2          |Dispute          |0.0102315

**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta q. Zonas con percentil 99 de duración/distancia fuera de rango (posible congestión/eventos)

In [27]:
q_q = (
    spark.read
    .format("snowflake")
    .options(**sfOptionsAnalytics)
    .option(
        "query",
        f"""
        SELECT
            pu_zone,
            APPROX_PERCENTILE(trip_duration_min, 0.99) AS p99_trip_duration_min,
            APPROX_PERCENTILE(trip_distance, 0.99) AS p99_trip_distance
        FROM {OBT_TABLE}
        WHERE pu_zone IS NOT NULL
          AND trip_duration_min IS NOT NULL
          AND trip_distance IS NOT NULL
          AND trip_duration_min BETWEEN 0 AND 300
          AND trip_distance BETWEEN 0 AND 100
        GROUP BY pu_zone
        ORDER BY p99_trip_duration_min DESC, p99_trip_distance DESC
        """
    )
    .load()
)

q_q.show(100, truncate=False)

+-----------------------------------+---------------------+-----------------+
|PU_ZONE                            |P99_TRIP_DURATION_MIN|P99_TRIP_DISTANCE|
+-----------------------------------+---------------------+-----------------+
|Arden Heights                      |168.967              |41.7914          |
|Eltingville/Annadale/Prince's Bay  |143.3928             |47.6876          |
|Far Rockaway                       |138.0993375          |30.943937778     |
|Hammels/Arverne                    |135.170564444        |31.776021053     |
|Rossville/Woodrow                  |129.3845             |46.854           |
|Rockaway Park                      |128.190625           |31.3816          |
|Heartland Village/Todt Hill        |124.7303             |39.4195          |
|Outside of NYC                     |120.008741513        |37.536259785     |
|Coney Island                       |119.200560653        |29.684336432     |
|Saint George/New Brighton          |118.97               |34.19

**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta r. Yield por milla (total_amount/trip_distance) por borough y hora

In [28]:
q_r = spark.sql("""
SELECT
    pu_borough,
    pickup_hour,
    AVG(CASE WHEN trip_distance > 0 THEN total_amount / trip_distance END) AS avg_yield_per_mile
FROM obt_trips
GROUP BY pu_borough, pickup_hour
ORDER BY pu_borough, pickup_hour
""")
q_r.show(200, truncate=False)

+-------------+-----------+------------------+
|pu_borough   |pickup_hour|avg_yield_per_mile|
+-------------+-----------+------------------+
|null         |0          |271.478832947     |
|null         |1          |258.319431593     |
|null         |2          |272.017652867     |
|null         |3          |257.12147482      |
|null         |4          |228.431320212     |
|null         |5          |135.860766607     |
|null         |6          |59.619125171      |
|null         |7          |70.558076693      |
|null         |8          |91.593378362      |
|null         |9          |106.454016176     |
|null         |10         |118.998462682     |
|null         |11         |130.450926293     |
|null         |12         |124.460714674     |
|null         |13         |104.463261744     |
|null         |14         |96.808749935      |
|null         |15         |117.14709919      |
|null         |16         |114.433437667     |
|null         |17         |146.483012382     |
|null        

**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta s. Cambios YoY en volumen y ticket promedio por service_type

In [29]:
q_s = spark.sql("""
WITH yearly AS (
    SELECT
        service_type,
        source_year,
        COUNT(*) AS total_trips,
        AVG(total_amount) AS avg_ticket
    FROM obt_trips
    GROUP BY service_type, source_year
)
SELECT
    service_type,
    source_year,
    total_trips,
    avg_ticket,
    LAG(total_trips) OVER (PARTITION BY service_type ORDER BY source_year) AS prev_year_trips,
    LAG(avg_ticket) OVER (PARTITION BY service_type ORDER BY source_year) AS prev_year_avg_ticket,
    ROUND(
        100.0 * (total_trips - LAG(total_trips) OVER (PARTITION BY service_type ORDER BY source_year))
        / NULLIF(LAG(total_trips) OVER (PARTITION BY service_type ORDER BY source_year), 0),
        2
    ) AS yoy_trips_pct,
    ROUND(
        100.0 * (avg_ticket - LAG(avg_ticket) OVER (PARTITION BY service_type ORDER BY source_year))
        / NULLIF(LAG(avg_ticket) OVER (PARTITION BY service_type ORDER BY source_year), 0),
        2
    ) AS yoy_avg_ticket_pct
FROM yearly
ORDER BY service_type, source_year
""")
q_s.show(100, truncate=False)

+------------+-----------+-----------+------------------+---------------+--------------------+-------------+------------------+
|service_type|source_year|total_trips|avg_ticket        |prev_year_trips|prev_year_avg_ticket|yoy_trips_pct|yoy_avg_ticket_pct|
+------------+-----------+-----------+------------------+---------------+--------------------+-------------+------------------+
|green       |2015       |18937057   |14.820283827629602|null           |null                |null         |null              |
|green       |2016       |16141551   |14.644182337868276|18937057       |14.820283827629602  |-14.76       |-1.19             |
|green       |2017       |11579087   |14.274499964461793|16141551       |14.644182337868276  |-28.27       |-2.52             |
|green       |2018       |8777350    |16.179663825642137|11579087       |14.274499964461793  |-24.20       |13.35             |
|green       |2019       |6128892    |18.311426352430423|8777350        |16.179663825642137  |-30.17    

**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Pregunta t. Días con alta congestion_surcharge: efecto en total_amount vs días normales

In [30]:
q_t = spark.sql("""
WITH daily AS (
    SELECT
        pickup_date,
        AVG(congestion_surcharge) AS avg_congestion_surcharge,
        AVG(total_amount) AS avg_total_amount
    FROM obt_trips
    GROUP BY pickup_date
),
classified AS (
    SELECT
        pickup_date,
        avg_congestion_surcharge,
        avg_total_amount,
        CASE
            WHEN avg_congestion_surcharge >= percentile_approx(avg_congestion_surcharge, 0.9) OVER ()
            THEN 'high_congestion'
            ELSE 'normal'
        END AS day_type
    FROM daily
)
SELECT
    day_type,
    COUNT(*) AS total_days,
    AVG(avg_total_amount) AS avg_total_amount
FROM classified
GROUP BY day_type
ORDER BY day_type
""")
q_t.show(truncate=False)

+---------------+----------+------------------+
|day_type       |total_days|avg_total_amount  |
+---------------+----------+------------------+
|high_congestion|259       |24.277789202539086|
|normal         |3812      |20.578034080890568|
+---------------+----------+------------------+



**Interpretación:** escribe aquí una breve conclusión con base en los resultados obtenidos.

## Conclusión general

Resume aquí los hallazgos más importantes del análisis:
- patrones de demanda
- diferencias entre yellow y green
- zonas más activas
- comportamiento de propinas
- efectos de congestión, duración y velocidad

In [31]:
print("Notebook 05 listo")

Notebook 05 listo
